In [ ]:
import numpy as np
def clone(dict):
    newdict={}
    for a in dict:
        newdict[a]=dict[a]
    return newdict

In [1324]:
all_chance_cards=[]
all_community_cards=[]
class prop():
    num_houses=0
    color_set='N/A'
    prop_type='color card' #Can be 'color card', 'railroad', or 'utility'
    cards_in_set=3
    cost=0
    house_cost=0
    mortgaged=False
    rents=[0,0,0,0,0,0]
    name=''
    def __init__(self,name,color_set,cards_in_set,cost,rents):
        self.num_houses=0
        self.name=name
        self.color_set=color_set
        self.prop_type='color card'
        self.cards_in_set=3
        if('rail' in color_set):
            self.prop_type='railroad'
        if('utility' in color_set):
            self.prop_type='utility'
        if(self.prop_type=='color card'):
            if(color_set in ['brown','black','pink','blue']):
                self.cards_in_set=2
            else:
                self.cards_in_set=3
        else:
            self.cards_in_set=2
        self.house_cost=0
        self.mortgaged=False
        if(color_set in ['brown','white','light blue']):
            self.house_cost=50
        if(color_set in ['black','purple','orange']):
            self.house_cost=100
        if(color_set in ['red', 'pink', 'yellow']):
            self.house_cost=150
        if(color_set in ['light green', 'green', 'blue']):
            self.house_cost=200
        self.rents=rents
        self.cost = cost
        if(self.prop_type=='railroad'):
            self.rents=[50,150]
    def display(self):
        print('Property:'+self.name)
        if(self.mortgaged):
            print('Property IS mortgaged')
            print('Price to unmortgage: $'+str(round(self.cost*0.55)))
        else:
            print('Property is NOT mortgaged')
            print('Money from mortgaging: $'+str(round(self.cost*0.5)))
            print('There are '+str(self.num_houses)+' houses on this property')
    def mortgage(self):
        if(not self.mortgaged):
            self.mortgaged=True
            print(self.name+ ' was MORTGAGED for $'+str(self.cost/2))
            return int(round(self.cost/2,3))
        return 0
    def unmortgage(self):
        if(self.mortgaged):
            self.mortgaged=False
            print(self.name+ ' was UNMORTGAGED for $'+str(1.1*self.cost/2))
            return -int(np.ceil(round((self.cost/2)*1.1,3)))
        return 0
    def build_house(self):
        if(self.num_houses<5):
            self.num_houses+=1
            print('A house was built on '+self.name+' for $'+str(self.house_cost)+'. There are now '+str(self.num_houses)+' houses here.')
            return -self.house_cost
        return 0
    def sell_house(self):
        if(self.num_houses>0):
            self.num_houses-=1
            print('A house was sold on '+self.name+' for $'+str(self.house_cost/2)+'. There are now '+str(self.num_houses)+' houses here.')
            return self.house_cost/2
        return 0
def load_props():
    #Board spaces indexed 0 to 55
    SPACE_DISPLAY_NAMES = [
    "Go", "Hilton Ave.", "Community Chest", "Aurora Ave.", "Income Tax", "Bartle Station", 
    "Graham St.", "Donaldson St.", "Valentine St.", "Donaldson Park Station", "Magnolia St.", 
    "Chance", "Benner St.", "Johnson St.", "Jail / Just Visiting", "First Ave.", "Raceway Gas", 
    "South Adelaide Ave.", "Community Chest", "Papa John's Station", "River Road", "Cedar Lane", 
    "Leia Lane", "Johnson Park Station", "Ella Lane", "Chance", "Gabriel Lane", "Ethan Lane", 
    "Free Parking", "Wayside Road", "Community Chest", "Cleveland Ave.", "Madison Ave.", 
    "HPMS Station", "Wayne St.", "HPHS", "Montgomery St.", "HP Public Library Station", 
    "South Park Ave.", "Brookfall Road", "Chance", "Marshall Drive", "Go to Jail", 
    "Highland Ave.", "Washington Ave.", "Community Chest", "Lexington Ave.", "Chef Ge's Station", 
    "Central Ave.", "Exeter St.", "Parker Road", "Dunkin' Station", "Chance", "Woodbridge", 
    "Luxury Tax", "Raritan"
    ]
    #Corresponding buy price, 99999 indicates unpurchaseable, go space is not in this list
    RAW_PURCHASE_PRICES = [
    60, 99999, 60, 99999, 200, 75, 75, 90, 200, 100, 99999, 100, 120, 99999, 120, 150, 
    120, 99999, 200, 140, 140, 160, 200, 180, 99999, 180, 200, 99999, 220, 99999, 220, 
    240, 200, 240, 150, 260, 200, 260, 260, 99999, 280, 99999, 280, 280, 99999, 300, 
    200, 300, 300, 320, 200, 99999, 350, 99999, 400
    ]

    #list of all rents by board index, excluding go
    DOLLAR_RENT_BY_RANK = [
    [2,0,4,200,50,5,5,6,50,6,0,6,8,0,8,5,10,0,50,10,10,12,50,14,0,14,16,0,18,0,18,20,50,20,5,22,50,22,22,0,24,0,24,24,0,26,50,26,26,28,50,0,35,100,50],
    [4,0,8,200,150,10,10,12,150,12,0,12,16,0,16,10,20,0,150,20,20,24,150,28,0,28,32,0,36,0,36,40,150,40,10,44,150,44,44,0,48,0,48,48,0,52,150,52,52,56,150,0,70,100,100],
    [10,0,20,200,150,25,25,30,150,30,0,30,40,0,35,10,35,0,150,50,50,60,150,70,0,70,80,0,90,0,90,100,150,90,10,100,150,110,110,0,120,0,200,200,0,225,150,130,130,150,150,0,175,100,200],
    [30,0,60,200,150,75,75,90,150,90,0,90,100,0,110,10,110,0,150,150,150,180,150,200,0,200,220,0,250,0,250,300,150,225,10,275,150,330,330,0,360,0,550,550,0,600,150,450,450,500,150,0,500,100,600],
    [90,0,180,200,150,160,160,190,150,270,0,270,300,0,350,10,350,0,150,400,400,450,150,500,0,500,550,0,700,0,700,750,150,550,10,750,150,800,800,0,850,0,900,900,0,975,150,1100,1100,1200,150,0,1100,100,1400],
    [160,0,320,200,150,280,280,320,150,400,0,400,450,0,475,10,500,0,150,575,575,650,150,700,0,700,750,0,875,0,875,925,150,750,10,1000,150,975,975,0,1025,0,1125,1125,0,1200,150,1400,1400,1500,150,0,1300,100,1700],
    [250,0,450,200,150,600,600,650,150,550,0,550,600,0,650,10,700,0,150,700,700,800,150,900,0,900,950,0,1050,0,1050,1100,150,950,10,1250,150,1150,1150,0,1200,0,1350,1350,0,1500,150,1700,1700,1800,150,0,1500,100,2000]
    ]

    #Data parsing
    RENTS_BY_INDEX = list(zip(*DOLLAR_RENT_BY_RANK))

    COLOR_GROUPS = [
        'brown', 'brown',
        'green rail',
        'white', 'white', 'white',
        'blue rail',
        'light blue', 'light blue', 'light blue',
        'black', 'black',
        'red rail',
        'purple', 'purple', 'purple',
        'brown rail',
        'orange', 'orange', 'orange', 
        'red', 'red', 'red',
        'green rail',
        'pink', 'pink',
        'blue rail',
        'yellow', 'yellow', 'yellow',
        'light green', 'light green', 'light green',
        'brown rail',
        'green', 'green', 'green',
        'red rail',
        'blue', 'blue'
    ]

    All_properties = []
    color_idx = 0

    for index, price in enumerate(RAW_PURCHASE_PRICES):
        if price != 99999:
            # +1 offset because SPACE_DISPLAY_NAMES[0] is 'Go'
            name = SPACE_DISPLAY_NAMES[index + 1]
            name_lower = name.lower()
            if 'gas' in name_lower or 'hphs' in name_lower:
                color_set = 'utility'
            else:
                color_set = COLOR_GROUPS[color_idx]
                color_idx += 1

            new_prop = prop(
                name=name,
                color_set=color_set,
                cards_in_set=0, #placeholder, will be set in prop.__init__
                cost=price,
                rents=list(RENTS_BY_INDEX[index])
            )
            All_properties.append(new_prop)

    #Output testing
    print(f"Successfully loaded {len(All_properties)} properties.\n")
    for p in All_properties[:42]:
        print(f"{p.name:<25} | Type: {p.prop_type:<10} | Set: {p.color_set:<10} | Cost: ${p.cost}")
    return All_properties

class game():
    bots=[]
    num_players=0
    property_list=[]
    def __init__(self,num_humans,bots=[]):
        bots=[]
        self.num_players=num_humans+len(bots)
        self.bots=bots
        self.property_list=load_props()
    def add_bot(self,bot):
        bot.parent_game=self
        self.bots+=[bot]
        self.num_players+=1
    def auction(self,prop,min_bid):
        bid=min_bid
        for bot in self.bots:
            newbid=bot.check_auction(prop,bid+5)
            if(not newbid == 'OUT'):
                bid=newbid
        print('Highest bid so far is $'+str(bid))
    def get_prop(self, name):
        #print('target name is'+name)
        for prop in self.property_list:
            #print(prop.name)
            if(prop.name==name):
                return prop
    def display(self):
        print('Displaying game info')
        print('Number of players: '+str(self.num_players))
        print('Bots:')
        for bot in self.bots:
            bot.display()
            
            print()
            print()
            print()
class card():
    card_name='Placeholder'
    action_statement='Does nothing'
    def __init__(self,name,statement):
        self.card_name=name
        self.action_statement=statement
    def display_card(self):
        print(self.card_name)
        print(self.action_statement)
def blank_card():
    return card('Blank','Blank')
default_preferences={'brown':1,'white':1,'light blue':1,'black':1,'purple':1,'orange':1,'red':1,'pink':1,'yellow':1,'light green':1,'green':1,'blue':1,
                      'blue rail':1, 'red rail':1,'green rail':1,'brown rail':1,'utility':1}
default_other_modifiers={'2 of 3':1, '3 of 3':1, '2 of 2':1, 'rail pair':1,'utility pair':1,'stuborness':1,'riskiness':1.5}
class bot():
    name=''
    parent_game='No game'
    Money=1500
    properties=[]
    chance=blank_card()
    community=blank_card()
    preference_modifiers={}#Preferences for each color set
    modifier_for_2_of_3=1
    modifier_for_3_of_3=1
    modifier_for_2_of_2=1
    modifier_for_rail_pair=1
    modifier_for_utility_pair=1
    supply_multipliers=clone(default_preferences)#Decreases when someone offers to give the bot a particular property
    demand_multipliers=clone(default_preferences)#Increases when someone asks the bot for a particular property
    stuborness=5
    safe_cash_threshold=100
    jail_turn=3 #0: Just got into jail on this turn. 1: completed 1st role 2: completed 2nd role 3 or higher: out of jail.
    riskiness=1.5
    def __init__(self, name, preference_modifiers,other_modifiers,game='No game'):
        self.name=name
        if(preference_modifiers=='default'):
            preference_modifiers=clone(default_preferences)
        if(other_modifiers=='default'):
            other_modifiers=clone(default_other_modifiers)
        self.preference_modifiers=clone(preference_modifiers)
        self.modifier_for_2_of_3=other_modifiers['2 of 3']
        self.modifier_for_3_of_3=other_modifiers['3 of 3']
        self.modifier_for_2_of_2=other_modifiers['2 of 2']
        self.modifier_for_rail_pair=other_modifiers['rail pair']
        self.modifier_for_utility_pair=other_modifiers['utility pair']
        self.stuborness=other_modifiers['stuborness']
        self.riskiness=other_modifiers['riskiness']
        self.parent_game=game
        if(not game=='No game'):
            game.add_bot(self)
        self.Money=1500
        self.properties=[]
        self.chance=blank_card()
        self.community=blank_card()
        supply_multipliers=clone(default_preferences)
        demand_multipliers=clone(default_preferences)
        self.safe_cash_threshold=100
        self.jail_turn=3
        #self.parent_game=game(0)
    def display(self):
        print('Bot:'+self.name)
        print('Money = $'+str(self.Money))
        print()
        print('Properties:')
        for p in self.properties:
            p.display()
            print()
        print()
        print('Chance Card:')
        self.chance.display_card()
        print()
        print('Community Chest Card:')
        self.community.display_card()
        print()
        if(self.jail_turn>2):
            print(self.name+' is not in jail')
        else:
            print(self.name+' has been in jail for '+str(self.jail_turn)+' turns.')
    def display_secret(self):
        print('Preference modifiers: ',self.preference_modifiers)
        print('Supply modifiers: ',self.supply_multipliers)
        print('Demand modifiers: ',self.demand_multipliers)
        print('Total value to self: ',self.get_total_value_to_self())
        print('Net worth: ',self.get_net_worth())
        print('Liquifiable cash: ',self.get_net_worth_without_selling())
        print('Risk modifier:',self.risk_modifier())
    def get_num_properties(self,color_set): #Returns the number of properties it has for that color set
        num=0
        for prop in self.properties:
            if(prop.color_set==color_set):
                num+=1
        return num
    
    def get_preference(self, prop): #Returns the bot's estimated value of a card that it has, modified to account for how close it is to having a monopoly
        #print('Running get_preference')
        if(type(prop)==type('  ')):
            prop=self.parent_game.get_prop(prop)
        #print(prop)
        num_props=self.get_num_properties(prop.color_set)
        max_num=prop.cards_in_set
        if(num_props==1):
            return self.preference_modifiers[prop.color_set]
        if(num_props==2 and prop.prop_type=='color card' and max_num==2):
            return self.preference_modifiers[prop.color_set]*self.modifier_for_2_of_2
        if(num_props==2 and prop.prop_type=='color card' and max_num==3):
            return self.preference_modifiers[prop.color_set]*self.modifier_for_2_of_3
        if(num_props==3 and prop.prop_type=='color card' and max_num==3):
            return self.preference_modifiers[prop.color_set]*self.modifier_for_3_of_3
        if(num_props==2 and prop.prop_type=='railroad'):
            return self.preference_modifiers[prop.color_set]*self.modifier_for_rail_pair
        if(num_props==2 and prop.prop_type=='utility'):
            return self.preference_modifiers[prop.color_set]*self.modifier_for_utility_pair
    
    """
    def check_use_card(self,situation='End of turn'):#Prints 'Used' if used (plus target if applicable). Prints 'Does not use card' if not used.
        #Situations are end of turn and beginning of turn (i.e. before dice roll). Maybe it should just ignore the beginning of the turn?
        if(situation=='End of turn'):

        if(situation=='Beginning of turn'):

    """
    def check_build(self,situation='End of turn'):
        monopolies=self.get_monopolies()
        #print(monopolies)
        newmono=[]
        for monopoly in monopolies:
            if(not ('rail' in monopoly or 'utility' in monopoly)):
                newmono+=[monopoly]
        monopolies=newmono
        if(len(monopolies)==0):
            return False
        max_houses=[]
        for monopoly in monopolies:
            num=0
            for prop in self.properties:
                if(prop.color_set==monopoly and prop.num_houses>num):
                    num=prop.num_houses
            max_houses+=[num]
        min_houses=[]
        for monopoly in monopolies:
            num=5
            for prop in self.properties:
                if(prop.color_set==monopoly and prop.num_houses<num):
                    num=prop.num_houses
            min_houses+=[num]
        highest_preference=0
        highest_num_houses=0
        index=-1
        i=0
        while(i<len(monopolies)):
            if((max_houses[i]>highest_num_houses or (max_houses[i]==highest_num_houses and self.preference_modifiers[monopolies[i]]>highest_preference)) and min_houses[i]!=5):
                highest_num_houses=max_houses[i]
                highest_preference=self.preference_modifiers[monopolies[i]]
                index=i
            i+=1
        if(index==-1):
            return False
        monopoly=monopolies[index]
        props=[]
        for prop in self.properties:
            if(prop.color_set==monopoly):
                props+=[prop]
        #print(props)
        #print(self.properties)
        cost=props[0].house_cost
        liquifiable_cash=self.get_net_worth_without_selling()
        if(highest_num_houses==0): #Correction so it doesn't think it can mortgage properties after building the first house
            s=0
            for prop in props:
                s+=prop.cost/2
            liquifiable_cash-=s
        """
        Now the bot has to decide whether to purchase another house on this monopoly, given the cost, risk modifier, and available cash.
        I'm leaning towards just putting in a minimum spare cash threshold.
        """
        done=False
        built_house=False
        while(liquifiable_cash-props[0].house_cost>=self.safe_cash_threshold and self.Money-props[0].house_cost>=0 and not done):
            min_houses=5
            props_with_min_houses=[]
            for prop in props:
                if(prop.num_houses<min_houses):
                    #print(prop.name)
                    #print(prop.num_houses)
                    #print(min_houses)
                    min_houses=prop.num_houses
                    props_with_min_houses=[prop]
                else:
                    if(prop.num_houses==min_houses):
                        props_with_min_houses+=[prop]
            prop_to_buy=props_with_min_houses[0]
            for prop in props_with_min_houses:
                if(prop.cost>prop_to_buy.cost):
                    prop_to_buy=prop
            self.Money+=prop_to_buy.build_house()
            built_house=True
            done=True
            for prop in props:
                if(prop.num_houses<5):
                    done=False
            liquifiable_cash=self.get_net_worth_without_selling()
        #print('built_house=',built_house)
        return built_house
    def check_jail_bailout(self):
        liquifiable_cash=self.get_net_worth_without_selling()
        if(liquifiable_cash>=self.safe_cash_threshold+50 and self.Money>=50 and self.safe_cash_threshold<1000):
            #print(self.name+' pays $50 to get out of jail, changing it\'s money from $'+str(self.Money)+' to $'+str(self.Money-50)+'.')
            self.Money-=50
            self.jail_turn=3
        else:
            #print(self.name+' does not pay to get out of jail. This will be it\'s roll #'+str(self.jail_turn)+' to escape')
            self.jail_turn+=1
        """
        Still have to figure out how to decide.
        """
    def free(self):
        self.jail_turn=3
        print(self.name + ' is no longer in jail.')
    def evaluate_trade(self, trade):
        #First, check if the trade is legal:
        cost=-trade.offered_money
        off_props=[]
        for prop in trade.offered_properties:
            if(type(prop)==type('  ')):
                off_props+=[self.parent_game.get_prop(prop)]
            else:
                off_props+=[prop]
        ask_props=[]
        for prop in trade.asked_for_properties:
            if(type(prop)==type('  ')):
                ask_props+=[self.parent_game.get_prop(prop)]
            else:
                ask_props+=[prop]
        i=0
        while(i<len(off_props)):
            if(off_props[i].mortgaged):
                cost+=off_props[i].cost/20
            i+=1
        if(cost>self.Money):
            print('trade rejected')
            return
        for prop in ask_props:
            found=False
            for my_prop in self.properties:
                if(my_prop.name==prop.name):
                    found=True
            if(not found):
                print('trade rejected')
                return
        net_value=trade.offered_money
        if(trade.will_give_asker_monopoly):
            net_value-=500
        #print(off_props)
        #print(ask_props)
        offered_colors=[]
        offered_props=[]
        tot_cost=[]
        for prop in off_props:
            if(prop.color_set in offered_colors):
                i=0
                while(i<len(offered_colors)):
                    if(offered_colors[i]==prop.color_set):
                        offered_props[i]+=[prop]
                        tot_cost[i]+=[prop.cost]
                    i+=1
            else:
                offered_colors+=[prop.color_set]
                offered_props+=[[prop]]
                tot_cost+=[prop.cost]
        #print(offered_colors)
        i=0
        while(i<len(offered_colors)):
            color=offered_colors[i]
            num_have=self.get_num_properties(color)
            num_will_have=len(offered_props[i])
            max_num=offered_props[i][0].cards_in_set
            prop_type=offered_props[i][0].prop_type
            pref=self.preference_modifiers[color]*self.supply_multipliers[color]
            cost_offered=tot_cost[i]
            cost_owned=self.cost_of_all_in_set(color)
            add=0
            if(prop_type=='utility'):
                if(num_will_have==2):
                    if(num_have==0):
                        add=cost_offered*self.modifier_for_utility_pair
                    else:
                        add=(cost_owned+cost_offered)*self.modifier_for_utility_pair-cost_owned
                if(num_will_have==1):
                    add=cost_offered
            if(prop_type=='railroad'):
                if(num_will_have==2):
                    if(num_have==0):
                        add=cost_offered*self.modifier_for_rail_pair
                    else:
                        add=(cost_owned+cost_offered)*self.modifier_for_rail_pair-cost_owned
                if(num_will_have==1):
                    add=cost_offered
            if(prop_type=='color card'):
                if(max_num==2):
                    if(num_will_have==2):
                        if(num_have==0):
                            add=cost_offered*self.modifier_for_2_of_2
                        else:
                            add=(cost_owned+cost_offered)*self.modifier_for_2_of_2-cost_owned
                    if(num_will_have==1):
                        add=cost_offered
                if(max_num==3):
                    if(num_will_have==3):
                        if(num_have==0):
                            add=cost_offered*self.modifier_for_3_of_3
                        if(num_have==1):
                            add=(cost_offered+cost_owned)*self.modifier_for_3_of_3-cost_owned
                        if(num_have==2):
                            add=(cost_offered+cost_owned)*self.modifier_for_3_of_3-cost_owned*self.modifier_for_2_of_3
                    if(num_will_have==2):
                        if(num_have==0):
                            add=cost_offered*self.modifier_for_2_of_3
                        else:
                            add=(cost_offered+cost_owned)*self.modifier_for_2_of_3-cost_owned
    
                    if(num_will_have==1):
                        add=cost_offered
            net_value+=add*pref
            i+=1
        #Now, it calculates how much value it is giving up.
        asked_colors=[]
        asked_props=[]
        tot_cost=[]
        for prop in ask_props:
            if(prop.color_set in asked_colors):
                i=0
                while(i<len(asked_colors)):
                    if(asked_colors[i]==prop.color_set):
                        asked_props[i]+=[prop]
                        tot_cost[i]+=[prop.cost]
                    i+=1
            else:
                asked_colors+=[prop.color_set]
                asked_props+=[[prop]]
                tot_cost+=[prop.cost]
        i=0
        while(i<len(asked_colors)):
            color=asked_colors[i]
            num_have=self.get_num_properties(color)
            num_will_have=num_have-len(asked_props[i])
            max_num=asked_props[i][0].cards_in_set
            prop_type=asked_props[i][0].prop_type
            pref=self.preference_modifiers[color]*self.demand_multipliers[color]
            cost_asked=tot_cost[i]
            cost_owned=self.cost_of_all_in_set(color)
            add=0
            if(prop_type=='utility'):
                if(num_will_have==0):
                    if(num_have==2):
                        add=-cost_asked*self.modifier_for_utility_pair
                    else:
                        add=-cost_asked
                if(num_will_have==1):
                    add=(cost_owned-cost_asked)-cost_owned*self.modifier_for_utility_pair
            if(prop_type=='railroad'):
                if(num_will_have==0):
                    if(num_have==2):
                        add=-cost_asked*self.modifier_for_rail_pair
                    else:
                        add=-cost_asked
                if(num_will_have==1):
                    add=cost_owned-cost_asked-cost_owned*self.modifier_for_rail_pair
            if(prop_type=='color card'):
                if(max_num==2):
                    if(num_will_have==0):
                        if(num_have==2):
                            add=-cost_asked*self.modifier_for_2_of_2
                        else:
                            add=-cost_asked
                    if(num_will_have==1):
                        add=cost_owned-cost_asked-cost_owned*self.modifier_for_2_of_2
                if(max_num==3):
                    if(num_will_have==0):
                        if(num_have==3):
                            add=-cost_asked*self.modifier_for_3_of_3
                        if(num_have==2):
                            add=-cost_asked*self.modifier_for_2_of_3
                        if(num_have==1):
                            add=-cost_asked
                    if(num_will_have==1):
                        if(num_have==3):
                            add=cost_owned-cost_asked-cost_owned*self.modifier_for_3_of_3
                        else:
                            add=cost_owned-cost_asked-cost_owned*self.modifier_for_2_of_3
                    if(num_will_have==2):
                        add=(cost_owned-cost_asked)*self.modifier_for_2_of_3-cost_owned*self.modifier_for_3_of_3
            net_value+=add*pref
            i+=1
        risk_mod=self.risk_modifier()
        liquifiable_cash=self.get_net_worth_without_selling()
        """
        Now, calculate the bot's total change in absolute networth from the trade.
        """
        net_worth_change=trade.offered_money
        #print(net_worth_change)
        for prop in off_props:
            if(not prop.mortgaged):
                net_worth_change+=prop.cost/2
            else:
                net_worth_change-=prop.cost/20
                net_value-=prop.cost/20+prop.cost/2
        #print(net_worth_change)
        for prop in ask_props:
            #prop.display()
            if(not prop.mortgaged):
                net_worth_change-=prop.cost/2
                #print('subtracting')
            else:
                net_value+=prop.cost/2
        #print(net_worth_change)
        """
        Here, make a decision about whether to accept based on the net_value change, net_worth_change, and risk_mod.
        """
        #Here is one attempt:
        #print(net_value)
        #print(risk_mod)
        #print(net_worth_change)
        true_value=net_value+(self.riskiness-risk_mod)*net_worth_change
        if(true_value>0):
            print('Trade accepted')
            self.accept_trade(trade)
        else:
            print('Trade rejected')
        """
        Finally, modify preferences and opponent preferences based on the trade offer
        """
        #My initial idea:
        for color in offered_colors:
            self.supply_multipliers[color]-=0.05
        for color in asked_colors:
            self.demand_multipliers[color]+=0.05
    def accept_trade(self, trade):
        self.modify_money(trade.offered_money)
        for prop in trade.offered_properties:
            if(type(prop)==type('  ')):
                prop=self.parent_game.get_prop(prop)
            self.properties+=[prop]
            if(prop.mortgaged):
                self.modify_money(-prop.cost/20)
        for prop in trade.asked_for_properties:
            if(type(prop)==type('  ')):
                prop=self.parent_game.get_prop(prop)
            newprops=[]
            i=0
            while(i<len(self.properties)):
                if(prop.name!=self.properties[i].name):
                    newprops+=[self.properties[i]]
                i+=1
            self.properties=newprops
    def check_buy(self, prop):
        prop=self.parent_game.get_prop(prop)
        net_value=0
        color=prop.color_set
        props=[]
        for p in self.properties:
            if(p.color_set==color):
                props+=[p]
        prop_type=prop.prop_type
        max_num=prop.cards_in_set
        num_have=len(props)
        cost_have=self.cost_of_all_in_set(color)
        cost=prop.cost
        if(prop_type=='utility'):
            if(num_have==0):
                net_value=cost
            if(num_have==1):
                net_value=(cost+cost_have)*self.modifier_for_utility_pair-cost_have
        if(prop_type=='railroad'):
            if(num_have==0):
                net_value=cost
            if(num_have==1):
                net_value=(cost+cost_have)*self.modifier_for_rail_pair-cost_have
        if(prop_type=='color card'):
            if(num_have==0):
                net_value=cost
            if(num_have==1 and max_num==2):
                net_value=(cost+cost_have)*self.modifier_for_2_of_2-cost_have
            if(num_have==1 and max_num==3):
                net_value=(cost+cost_have)*self.modifier_for_2_of_3-cost_have
            if(num_have==2):
                net_value=(cost+cost_have)*self.modifier_for_3_of_3-cost_have*self.modifier_for_2_of_3
        net_value=net_value*self.preference_modifiers[color]*self.risk_modifier()
        if(net_value>=cost and cost<=self.Money):
            print(self.name+' buys '+prop.name+' for $'+str(cost)+', changing it\'s money from $'+str(self.Money)+' to $'+str(self.Money-cost)+'.')
            self.Money-=cost
            self.properties+=[prop]
        else:
            print(self.name+' auctions '+prop.name+'.')
            self.check_auction(prop,1)
    def buy(self,prop,cost):#For use when buying from an auction
        prop=self.parent_game.get_prop(prop)
        print(self.name+' buys '+prop.name+' for $'+str(cost)+', changing it\'s money from $'+str(self.Money)+' to $'+str(self.Money-cost)+'.')
        self.Money-=cost
        self.properties+=[prop]
    def check_auction(self, prop, min_bid): #Returns 'OUT' or the bid amount
        if(type(prop)==type('  ')):
            prop=self.parent_game.get_prop(prop)
        value=0
        color=prop.color_set
        prop_type=prop.prop_type
        if(prop_type=='color card'):
            num=self.get_num_properties(color)
            max_num=prop.cards_in_set
            if(num==0):
                value=self.preference_modifiers[color]*prop.cost
            if(num==1 and max_num==2):
                value=self.preference_modifiers[color]*((prop.cost+self.cost_of_all_in_set(color))*self.modifier_for_2_of_2-self.cost_of_all_in_set(color))
            if(num==1 and max_num==3):
                value=self.preference_modifiers[color]*((prop.cost+self.cost_of_all_in_set(color))*self.modifier_for_2_of_3-self.cost_of_all_in_set(color))
            if(num==2 and max_num==3):
                value=self.preference_modifiers[color]*((prop.cost+self.cost_of_all_in_set(color))*self.modifier_for_3_of_3-self.cost_of_all_in_set(color)*self.modifier_for_2_of_3)
        if(prop_type=='railroad'):
            num=self.get_num_properties(color)
            if(num==0):
                value=self.preference_modifiers[color]*prop.cost
            if(num==1):
                value=self.preference_modifiers[color]*((prop.cost+self.cost_of_all_in_set(color))*self.modifier_for_rail_pair-self.cost_of_all_in_set(color))
        if(prop_type=='utility'):
            num=self.get_num_properties(color) #Returns 0 or 1 based on whether the bot has the other utility
            if(num==0):
                value=self.preference_modifiers[color]*prop.cost
            if(num==1):
                value=self.preference_modifiers[color]*((prop.cost+self.cost_of_all_in_set(color))*self.modifier_for_utility_pair-self.cost_of_all_in_set(color))
        #Value of card to bot is now set.
        max_bid=value*self.risk_modifier()
        if(max_bid<min_bid):
            print(self.name+' is out.')
            return 'OUT'
        else:
            bid=int(np.random.rand()*(max_bid+1-min_bid)+min_bid)
            print(self.name+' bids $'+str(bid))
            return bid
    def get_monopolies(self):#returns all the monopolies that this player has
        colors=[]
        properties=[]
        for prop in self.properties:
            if(prop.color_set in colors):
                i=0
                index=-1
                while(i<len(colors)):
                    if(colors[i]==prop.color_set):
                        index=i
                    i+=1
                properties[index]+=[prop]
            else:
                colors+=[prop.color_set]
                properties+=[[prop]]
        monopolies=[]
        i=0
        while(i<len(colors)):
            if(len(properties[i])==properties[i][0].cards_in_set):
                monopolies+=[colors[i]]
            i+=1
        return monopolies
    def risk_modifier(self):
        networth=self.get_net_worth()
        total_value_to_self=self.get_total_value_to_self()
        """
        Returns a modifier to multiply how much money the bot is willing to spend on a given transaction, based on how much spare cash it has.
        My idea is that it should go to 0 if total_value_to_self is a lot larger than its networth, i.e. if it doesn't have much cash to liquify but
        already has a lot of useful stuff. It should probably go to 1 as networth --> total_value_to_self, indicating it mostly just has unused cash.
        """
        return self.riskiness*networth/total_value_to_self
    def cost_of_all_in_set(self,colorset): #returns the sum of the costs of all the cards the bot has in a given color set. Useful for some calculations
        s=0
        i=0
        while(i<len(self.properties)):
            if(self.properties[i].color_set==colorset):
                s+=self.properties[i].cost
            i+=1
        return s
    def modify_money(self,change):
        newmoney=self.Money+change
        if(newmoney>=0):
            print(self.name+'\'s cash changed from $'+str(self.Money)+' to $'+str(newmoney))
            self.Money=newmoney
            return
        #Now it mortgages its properties, starting with the one it deems least valuable
        ratios=[]
        i=0
        while(i<len(self.properties)):
            prop=self.properties[i]
            if(prop.num_houses==0 and prop.mortgaged==False):
                preference=self.get_preference(prop)
                ratios+=[preference]
            else:
                ratios+=[10**9] #10**9 is a placeholder to denote that this property cannot be mortgaged
            i+=1
        index_order=[]
        while(len(index_order)<len(ratios)):
            m=-10**9
            index=-1
            i=0
            while(i<len(ratios)):
                if(not(i in index_order) and ratios[i]>m):
                    index=i
                    m=ratios[i]
                i+=1
            index_order+=[index]
        i=0
        while(self.Money+change<0 and i<len(index_order)):
            index=index_order[len(index_order)-1-i]
            if(ratios[index]<10**9):
                self.Money+=self.properties[index].mortgage()
            i+=1
        if(self.Money+change>=0):
            self.modify_money(change)
            return
        #Now, it sells houses
        monopolies_with_houses=[]
        i=0
        while(i<len(self.properties)):
            if(self.properties[i].num_houses>0):
                monopolies_with_houses+=[self.properties[i].color_set]
            i+=1
        monopolies_with_houses=list(set(monopolies_with_houses))
        #print(monopolies_with_houses)
        ps=[]
        for monopoly in monopolies_with_houses:
            p=''
            for prop in self.parent_game.property_list:
                if(prop.color_set==monopoly):
                    p=prop
            ps+=[p]
        monopolies_in_order=[]
        while(len(monopolies_in_order)<len(monopolies_with_houses)):
            m=-10**9
            monopoly=''
            i=0
            while(i<len(monopolies_with_houses)):
                monopoly=monopolies_with_houses[i]
                #print(monopolies_with_houses[i])
                if(not(monopolies_with_houses[i] in monopolies_in_order) and self.get_preference(ps[i])>m):
                    m=self.get_preference(ps[i])
                i+=1
            monopolies_in_order+=[monopoly]
        while(self.Money+change<0 and len(monopolies_in_order)>0):
            monopoly=monopolies_in_order[-1]
            monopolies_in_order=monopolies_in_order[:-1]
            props=[]
            i=0
            while(i<len(self.properties)):
                if(self.properties[i].color_set==monopoly):
                    props+=[self.properties[i]]
                i+=1
            done=False
            while(self.Money+change<0 and not done):
                max_num_houses=0
                i=0
                while(i<len(props)):
                    num=props[i].num_houses
                    if(num>max_num_houses):
                        max_num_houses=num
                    i+=1
                if(max_num_houses==0):
                    done=True
                else:
                    sell_prop=''
                    cost=0
                    i=0
                    while(i<len(props)):
                        if(props[i].num_houses==max_num_houses and props[i].cost>cost):
                            sell_prop=props[i]
                        i+=1
                    self.Money+=sell_prop.sell_house()
            while(self.Money+change<0 and len(props)>0):
                index=-1
                cost=10**9
                i=0
                while(i<len(props)):
                    if(props[i].cost<cost):
                        index=i
                        cost=props[i].cost
                    i+=1
                self.Money+=props[index].mortgage()
                newprops=[]
                i=0
                while(i<len(props)):
                    if(not i==index):
                        newprops+=[props[i]]
                    i+=1
                props=newprops
        if(self.Money+change>=0):
            self.modify_money(change)
        else:
            print(self.name+' declares bankruptcy.')
    def pay_rent(self, rent):
        self.modify_money(-rent)
        if(rent>self.safe_cash_threshold):
            self.safe_cash_threshold=rent
    def mortgage(self, prop_name):
        prop=''
        i=0
        while(i<len(self.properties)):
            if(self.properties[i].name==prop_name and self.properties[i].mortgaged==False):
                prop=self.properties[i]
            i+=1
        if(prop!=''):
            self.Money+=prop.mortgage
        else:
            print('Error. '+prop_name+' could not be mortgaged because '+self.name+' does not own it or it is already mortgaged.')
    def get_net_worth(self): #How much total money can be liquidated
        worth=0
        worth+=self.Money
        for prop in self.properties:
            worth+=prop.num_houses*prop.house_cost*0.5
            if(not prop.mortgaged):
                worth+=prop.cost/2
        return worth
    def get_total_value_to_self(self):
        worth=0
        worth+=self.Money
        for prop in self.properties:
            worth+=prop.num_houses*prop.house_cost
            if(prop.mortgaged):
                worth+=self.get_preference(prop)*prop.cost-prop.cost/2
            else:
                worth+=self.get_preference(prop)*prop.cost
        return worth
    def get_net_worth_without_selling(self):
        worth=0
        worth+=self.Money
        for prop in self.properties:
            if(prop.num_houses==0 and not prop.mortgaged):
                worth+=prop.cost/2
        return worth
    def deflate_trade_modifiers():
        for key in self.supply_multiplier:
            if(self.supply_multipliers[key]<1):
                self.supply_multipliers[key]+=0.05/stuborness
                if(self.supply_multipliers[key]>1):
                    self.supply_multipliers[key]=1
        for key in self.demand_multipliers:
            if(self.demand_multipliers[key]>1):
                self.demand_multipliers[key]-=0.05/stuborness
                if(self.demand_multipliers[key]>1):
                    self.demand_multipliers[key]=1
    def process_end_of_turn(self):
        #self.check_use_card()
        done=False
        while(not done):
            b=self.check_build()
            print('b=',b)
            done=not b
            print('done=',done)
        self.display_assets()
        self.deflate_trade_modifiers()
        if(self.safe_cash_threshold<2000):
            self.safe_cash_threshold+=20
    def process_beginning_of_turn(self):
        #self.check_use_card(situation='Beginning of turn')
        self.check_jail_bailout()




    
class trade_offer():
    offered_properties=[]
    #offered_cards=[]
    offered_money=0 #Negative if you are offering the bot money
    asked_for_properties=[]
    #asked_for_cards=[]
    will_give_asker_monopoly=False
    def __init__(self,off_props,off_money,asked_props,will_give_asker_monopoly):
        if(type(off_props)!=type([])):
            off_props=[off_props]
        if(type(asked_props)!=type([])):
            asked_props=[asked_props]
        self.offered_properties=off_props
        self.asked_for_properties=asked_props
        self.offered_money=off_money
        self.will_give_asker_monopoly=will_give_asker_monopoly

In [1316]:
mybot.display_secret() #For debugging only

Preference modifiers:  {'brown': 1, 'white': 1, 'light blue': 1, 'black': 1, 'purple': 1, 'orange': 1, 'red': 1, 'pink': 1, 'yellow': 1, 'light green': 1, 'green': 1, 'blue': 1, 'blue rail': 1, 'red rail': 1, 'green rail': 1, 'brown rail': 1, 'utility': 1}
Supply modifiers:  {'brown': 1, 'white': 1, 'light blue': 1, 'black': 1, 'purple': 1, 'orange': 1, 'red': 1, 'pink': 1, 'yellow': 1, 'light green': 1, 'green': 1, 'blue': 1, 'blue rail': 1, 'red rail': 1, 'green rail': 1, 'brown rail': 1, 'utility': 1}
Demand modifiers:  {'brown': 1, 'white': 1, 'light blue': 1, 'black': 1, 'purple': 1, 'orange': 1.4500000000000004, 'red': 1, 'pink': 1, 'yellow': 1, 'light green': 1, 'green': 1, 'blue': 1, 'blue rail': 1, 'red rail': 1, 'green rail': 1, 'brown rail': 1, 'utility': 1}
Total value to self:  1600
Net worth:  1420.0
Liquifiable cash:  1420.0
Risk modifier: 1.33125


In [1326]:
#Code to create new game
ourgame=game(6)
my_preferences={'brown':1,'white':1,'light blue':1,'black':1,'purple':1,'orange':1,'red':1,'pink':1,'yellow':1,'light green':1,'green':1,'blue':1,
                      'blue rail':1, 'red rail':1,'green rail':1,'brown rail':1,'utility':1}
my_other_modifiers={'2 of 3':1, '3 of 3':1, '2 of 2':1, 'rail pair':1,'utility pair':1,'stuborness':1,'riskiness':1.5}
mybot=bot('My bot',my_preferences,my_other_modifiers,game=ourgame)

Successfully loaded 42 properties.

Hilton Ave.               | Type: color card | Set: brown      | Cost: $60
Aurora Ave.               | Type: color card | Set: brown      | Cost: $60
Bartle Station            | Type: railroad   | Set: green rail | Cost: $200
Graham St.                | Type: color card | Set: white      | Cost: $75
Donaldson St.             | Type: color card | Set: white      | Cost: $75
Valentine St.             | Type: color card | Set: white      | Cost: $90
Donaldson Park Station    | Type: railroad   | Set: blue rail  | Cost: $200
Magnolia St.              | Type: color card | Set: light blue | Cost: $100
Benner St.                | Type: color card | Set: light blue | Cost: $100
Johnson St.               | Type: color card | Set: light blue | Cost: $120
First Ave.                | Type: color card | Set: black      | Cost: $120
Raceway Gas               | Type: utility    | Set: utility    | Cost: $150
South Adelaide Ave.       | Type: color card | Set: black

In [1328]:
ourgame.display() #Checks status of game

Displaying game info
Number of players: 7
Bots:
Bot:My bot
Money = $1500

Properties:

Chance Card:
Blank
Blank

Community Chest Card:
Blank
Blank

My bot is not in jail





In [1330]:
mybot.process_beginning_of_turn()

In [1300]:
mybot.check_buy('Property name')#Use when the bot lands on a property

My bot buys Gabriel Lane for $180, changing it's money from $1500 to $1320.
My bot buys Ella Lane for $180, changing it's money from $1320 to $1140.
My bot buys Ethan Lane for $200, changing it's money from $1140 to $940.


In [1314]:
mybot.evaluate_trade(trade_offer([],300,['Ethan Lane'],False))

Trade accepted
My bot's cash changed from $940 to $1240


In [443]:
ourgame.auction('Raritan',432)

My bot bids $443
Highest bid so far is $443


In [241]:
mybot.buy('Raritan',59)

My bot buys Hilton Ave. for $59, changing it's money from $450 to $391.


In [1019]:
mybot.check_build()

A house was built on Ethan Lane for $100. There are now 1 houses here.
A house was built on Gabriel Lane for $100. There are now 1 houses here.
A house was built on Ella Lane for $100. There are now 1 houses here.
A house was built on Ethan Lane for $100. There are now 2 houses here.
A house was built on Gabriel Lane for $100. There are now 2 houses here.


True

In [1021]:
mybot.modify_money(-750)

Brookfall Road was MORTGAGED for $130.0
Aurora Ave. was MORTGAGED for $30.0
Hilton Ave. was MORTGAGED for $30.0
A house was sold on Ethan Lane for $50.0. There are now 1 houses here.
A house was sold on Gabriel Lane for $50.0. There are now 1 houses here.
A house was sold on Ethan Lane for $50.0. There are now 0 houses here.
A house was sold on Ella Lane for $50.0. There are now 0 houses here.
A house was sold on Gabriel Lane for $50.0. There are now 0 houses here.
Gabriel Lane was MORTGAGED for $90.0
Ella Lane was MORTGAGED for $90.0
Ethan Lane was MORTGAGED for $100.0
My bot's cash changed from $780.0 to $30.0
